In [1]:
import pandas as pd
from pathlib import Path

# Load the raw data file from the data directory
csv_path = Path('..') / 'data' / 'raw' / 'healthcare-analytics-patient-flow-data' / 'healthcare_analytics_patient_flow_data.csv'
data = pd.read_csv(csv_path)

# Display the first few rows
data.head()

,Patient Id,Patient Admission Date,Patient Admission Time,Merged,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime
0,780-96-6113,9/9/2024,9:25:00 AM,W. Breede,Female,63,African American,NaN,Not Admission,5.0,32
1,714-35-6722,9/9/2024,4:42:00 PM,Y. Baldetti,Male,31,Asian,Orthopedics,Not Admission,NaN,22
2,571-85-3714,9/9/2024,12:14:00 AM,M. Semerad,Male,75,White,General Practice,Not Admission,NaN,16
3,404-43-9499,9/9/2024,8:33:00 PM,K. Blaydes,Male,79,African American,General Practice,Admission,NaN,38
4,552-51-5855,9/9/2024,7:25:00 PM,F. Dickerson,Female,24,African American,NaN,Admission,NaN,36


In [2]:
patient_subset = data[["Patient Id", "Patient Admission Date", "Patient Admission Time"]]
patient_subset.head()


,Patient Id,Patient Admission Date,Patient Admission Time
0,780-96-6113,9/9/2024,9:25:00 AM
1,714-35-6722,9/9/2024,4:42:00 PM
2,571-85-3714,9/9/2024,12:14:00 AM
3,404-43-9499,9/9/2024,8:33:00 PM
4,552-51-5855,9/9/2024,7:25:00 PM


In [3]:
patient_subset.describe()

,Patient Id,Patient Admission Date,Patient Admission Time
count,9216,9216,9216
unique,9216,579,1437
top,780-96-6113,12/4/2023,9:34:00 AM
freq,1,30,16


In [4]:
# Create a combined datetime column from the date and time fields.
# The raw data uses day-first values like "31/12/2023 11:05:00 PM".
patient_subset["timestamp"] = pd.to_datetime(
    patient_subset["Patient Admission Date"].astype(str).str.strip()
    + " "
    + patient_subset["Patient Admission Time"].astype(str).str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="coerce"
)

valid_timestamps = patient_subset["timestamp"].dropna()

# Compute summary metrics
unique_days = valid_timestamps.dt.date.nunique()
smallest_timestamp = valid_timestamps.min()
largest_timestamp = valid_timestamps.max()

print(f"Number of unique days: {unique_days}")
print(f"Smallest timestamp: {smallest_timestamp}")
print(f"Largest timestamp: {largest_timestamp}")

Number of unique days: 579
Smallest timestamp: 2023-04-01 01:13:00
Largest timestamp: 2024-10-30 23:44:00


In [5]:
# Show rows where the admission date/time could not be parsed
invalid_timestamp_rows = patient_subset.loc[
    patient_subset["timestamp"].isna()
].copy()

print(f"Rows with invalid timestamps: {len(invalid_timestamp_rows)}")

display(invalid_timestamp_rows)

Rows with invalid timestamps: 0


,Patient Id,Patient Admission Date,Patient Admission Time,timestamp


In [6]:
from pathlib import Path

output_dir = Path.home() / "programming" / "arrival_analysis" / "data"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "patient_subset.csv"
patient_subset.to_csv(output_path, index=False)

In [7]:
df = pd.read_csv(output_path)

In [8]:
dd_df = pd.read_csv("/home/rajiv/programming/kmds-dataset-util/data/raw/healthcare-analytics-patient-flow-data/data_dictionary_raw.csv")
dd_df.head()

,attribute,description,python_type
0,Patient Id,Unique anonymized patient identifier,str
1,Patient Admission Date,Date of patient arrival/registration (MM/DD/YYYY),datetime.date
2,Patient Admission Time,Time of patient arrival (HH:MM:SS AM/PM),pd.Timestamp
3,Patient_Admission_DateTime,Data quality issue: Appears to be corrupted na...,pd.Timestamp
4,Patient Gender,Biological sex: Male/Female,str


In [9]:
# Create a data quality report for the loaded dataframe
quality_report = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "row_count": len(df),
    "missing_cells": df.isna().sum().values,
    "missing_percent": (df.isna().mean() * 100).round(2).values,
    "unique_values": df.nunique(dropna=True).values
})

total_nan_cells = int(df.isna().sum().sum())

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Total NaN cells: {total_nan_cells}")

quality_report

Rows: 9216
Columns: 4
Duplicate rows: 0
Total NaN cells: 0


,column,data_type,row_count,missing_cells,missing_percent,unique_values
0,Patient Id,str,9216,0,0.0,9216
1,Patient Admission Date,str,9216,0,0.0,579
2,Patient Admission Time,str,9216,0,0.0,1437
3,timestamp,str,9216,0,0.0,9176


In [10]:
from pathlib import Path
import yaml

from data_prep.config import load_config

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

config_path = project_root / "configs" / "healthcare-prep.yaml"

# Build the raw and analysis dictionaries from the earlier data dictionary read
raw_dd = dd_df.copy()

# Normalize the raw attribute names so they match the actual dataset columns.
def normalize_attr(name):
    return str(name).replace("Merged ", "").strip()

raw_dd["attribute"] = raw_dd["attribute"].map(normalize_attr)

# Keep only the columns used in downstream analysis.
analysis_columns = []
for col in data.columns:
    norm = normalize_attr(col)
    if norm not in {"Patient Id", "Patient Admission Date", "Patient Admission Time"}:
        analysis_columns.append(norm)

analysis_dd = raw_dd[raw_dd["attribute"].isin(analysis_columns)].copy().reset_index(drop=True)

config_payload = {
    "raw_data_dictionary": raw_dd.to_dict(orient="records"),
    "analysis_data_dictionary": analysis_dd.to_dict(orient="records"),
    "analysis_columns": analysis_dd["attribute"].tolist(),
    "target_column": "Patient Admission Flag",
}

try:
    config = load_config(config_path)
    config.update(config_payload)
    with config_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)
except FileNotFoundError:
    config = config_payload
    with config_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

analysis_dd


,attribute,description,python_type
0,Patient Gender,Biological sex: Male/Female,str
1,Patient Age,Patient age in years (range: 0-79),int
2,Patient Race,Self-reported ethnicity/background,str
3,Department Referral,Specialty department for consultation,str
4,Patient Admission Flag,"Target Variable: Admission decision: ""Admissio...",bool
5,Patient Satisfaction Score,Post-visit satisfaction rating (0=very dissati...,float


In [ ]:
from pathlib import Path
import yaml
import pandas as pd

from data_prep.config import bootstrap_config, load_config
from data_prep.prepare import prepare_dataset


def prepare_healthcare_dataset(dataframe, *, prepared_dataset_dir, dataset_name="healthcare_dataset", output_format="csv", metadata=None):
    """Notebook-local healthcare preparation logic."""
    cleaned = dataframe.copy()
    required_columns = ["Patient Id", "Patient Admission Date", "Patient Admission Time"]
    missing = [col for col in required_columns if col not in cleaned.columns]
    if missing:
        raise ValueError(f"Healthcare dataset is missing required columns: {missing}")

    prepared = cleaned[required_columns].copy()
    prepared["timestamp"] = pd.to_datetime(
        prepared["Patient Admission Date"].astype(str).str.strip()
        + " "
        + prepared["Patient Admission Time"].astype(str).str.strip(),
        format="%d/%m/%Y %I:%M:%S %p",
        errors="coerce",
    )

    result_metadata = {
        "source": "healthcare-analytics-patient-flow-data",
        "preparation_logic": "patient_admission_time_serialization",
    }
    if metadata:
        result_metadata.update(metadata)

    return prepare_dataset(
        dataset_name=dataset_name,
        representation="tabular",
        task_characterization="classification",
        staging_dir="./data/raw/healthcare-analytics-patient-flow-data",
        prepared_dataset_dir=prepared_dataset_dir,
        data=prepared,
        output_format=output_format,
        filename=f"{dataset_name}_patient_subset",
        metadata=result_metadata,
    )


project_root = Path.cwd().resolve().parent
prepared_dir = project_root / "data" / "prepared" / "healthcare-analytics-patient-flow-data"
prepared_dir.mkdir(parents=True, exist_ok=True)

config_path = bootstrap_config(
    config_dir=project_root / "configs",
    config_name="healthcare-prep.yaml",
    dataset_name="healthcare-analytics-patient-flow-data",
    representation="tabular",
    task_characterization="time_series",
    staging_dir="./data/raw/healthcare-analytics-patient-flow-data",
    prepared_dataset_dir=str(prepared_dir),
    write=True,
)

prepare_result = prepare_healthcare_dataset(
    data,
    prepared_dataset_dir=prepared_dir,
    dataset_name="healthcare-analytics-patient-flow-data",
    output_format="csv",
    metadata={
        "raw_data_dictionary": raw_dd,
        "analysis_data_dictionary": analysis_dd,
        "data_dictionary": analysis_dd,
    },
)

# Keep both dictionary files in the staging area; the processed dictionary is the one used for analysis.
config = load_config(config_path)
config["raw_data_dictionary_path"] = str(project_root / "data" / "raw" / "healthcare-analytics-patient-flow-data" / "data_dictionary_raw.csv")
config["data_dictionary_path"] = str(Path(prepare_result["dictionary_files"]["data_dictionary"]))
config["processed_data_dictionary_path"] = config["data_dictionary_path"]
config["dictionary_files"] = prepare_result.get("dictionary_files", {})
with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(f"Bootstrap config: {config_path}")
print(f"Prepared dataset: {prepare_result['output_path']}")
print(f"Raw dictionary path: {config['raw_data_dictionary_path']}")
print(f"Processed dictionary path: {config['data_dictionary_path']}")
print(f"Dictionary files: {prepare_result.get('dictionary_files', {})}")
print(f"Files in prepared folder: {sorted(p.name for p in prepared_dir.iterdir())}")

pd.read_csv(prepare_result['output_path']).head()


Bootstrap config: /home/rajiv/programming/kmds-dataset-util/configs/healthcare-prep.yaml
Prepared dataset: /home/rajiv/programming/kmds-dataset-util/data/prepared/healthcare-analytics-patient-flow-data/healthcare-analytics-patient-flow-data_patient_subset.csv
Prepared dictionary: /home/rajiv/programming/kmds-dataset-util/data/prepared/healthcare-analytics-patient-flow-data/data_dictionary.csv
Dictionary files: {'data_dictionary': '/home/rajiv/programming/kmds-dataset-util/data/prepared/healthcare-analytics-patient-flow-data/data_dictionary.csv', 'raw_data_dictionary': '/home/rajiv/programming/kmds-dataset-util/data/prepared/healthcare-analytics-patient-flow-data/data_dictionary_raw.csv', 'analysis_data_dictionary': '/home/rajiv/programming/kmds-dataset-util/data/prepared/healthcare-analytics-patient-flow-data/data_dictionary_analysis.csv'}
Dictionary file exists: True
Files in prepared folder: ['data_dictionary.csv', 'data_dictionary_analysis.csv', 'data_dictionary_raw.csv', 'healthcar

,Patient Id,Patient Admission Date,Patient Admission Time,timestamp
0,780-96-6113,9/9/2024,9:25:00 AM,2024-09-09 09:25:00
1,714-35-6722,9/9/2024,4:42:00 PM,2024-09-09 16:42:00
2,571-85-3714,9/9/2024,12:14:00 AM,2024-09-09 00:14:00
3,404-43-9499,9/9/2024,8:33:00 PM,2024-09-09 20:33:00
4,552-51-5855,9/9/2024,7:25:00 PM,2024-09-09 19:25:00
